In [1]:
# data fetching
import pandas as pd
from sqlalchemy import create_engine, text
from urllib.parse import quote_plus
host="127.0.0.1"        # Change to server IP if remote
user="for_telematics"        # Your MySQL username
password="Arundv@02"# Your MySQL password
database="vehicle_telematics" 
port=3306 # Optional: specify your target database
print(repr(password))
password_encoded = quote_plus(password)
print(repr(password_encoded))
connection_string = f"mysql+pymysql://{user}:{password_encoded}@{host}:{port}/{database}"
engine = create_engine(connection_string)
with engine.connect() as connection:
# Load the entire table automatically into a structured DataFrame
    d_vehicles = pd.read_sql("SELECT * FROM vehicles", con=connection)
    d_drivers = pd.read_sql("SELECT * FROM drivers", con=connection)
    d_trips = pd.read_sql("SELECT * FROM trips", con=connection)
    d_maintenance = pd.read_sql("SELECT * FROM maintenance_logs", con=connection)
    d_sensor_events = pd.read_sql("SELECT * FROM sensor_events", con=connection)
print(d_trips.head())


'Arundv@02'
'Arundv%4002'
   trip_id  vehicle_id  driver_id          start_time            end_time  \
0        1          20          7 2026-06-07 03:35:57 2026-06-07 04:34:13   
1        2          16          1 2026-06-03 04:07:43 2026-06-03 04:34:54   
2        3           3          3 2026-08-02 08:55:34 2026-08-02 10:27:38   
3        4          18          6 2026-06-18 01:05:05 2026-06-18 02:04:46   
4        5          23         10 2026-07-01 01:10:38 2026-07-01 01:31:48   

   start_lat  start_long    end_lat   end_long  distance_km  fuel_used_liters  \
0  12.991227   80.393267  13.196871  80.383806        28.50              2.34   
1  13.185057   80.360339  13.030677  80.151220        19.58              1.38   
2  12.966271   80.341133  13.154547  80.170265        49.33              4.36   
3  13.111509   80.396203  13.198623  80.339691        33.01              2.57   
4  13.123208   80.253970  13.087450  80.168269        15.83              1.65   

   idle_minutes  
0     

In [61]:
# d_trips.isnull().sum()
d_trips=d_trips.dropna(subset=["fuel_used_liters"])
# print(d_sensor_events['speed_at_event'].describe())
d_sensor_events=d_sensor_events[d_sensor_events['speed_at_event']<120]
d_trips = d_trips[d_trips['end_time'] >= d_trips['start_time']]
d_trips = d_trips[d_trips['distance_km'] > 0]
d_vehicles['fuel_type'] = d_vehicles['fuel_type'].str.strip().str.title()
d_drivers['driver_name'] = d_drivers['driver_name'].str.strip()
d_sensor_events = d_sensor_events[d_sensor_events['trip_id'].isin(d_trips['trip_id'])]
orphaned_sensor_events = d_sensor_events[~d_sensor_events['trip_id'].isin(d_trips['trip_id'])]
d_vehicles["purchase_date"]=pd.to_datetime(d_vehicles["purchase_date"], errors='coerce')
d_drivers["join_date"]=pd.to_datetime(d_drivers["join_date"], errors='coerce')
print(d_vehicles['fuel_type'].unique())
print(d_vehicles['vehicle_type'].unique())
print(d_sensor_events['event_type'].unique())


<StringArray>
['Cng', 'Diesel', 'Petrol']
Length: 3, dtype: str
<StringArray>
['Mini Truck', 'Van', 'Tempo', 'Pickup']
Length: 4, dtype: str
<StringArray>
['harsh_cornering', 'harsh_braking', 'speeding', 'harsh_acceleration']
Length: 4, dtype: str


In [62]:
d_vehicles.to_sql('vehicles_clean', engine, if_exists='replace', index=False)
d_drivers.to_sql('drivers_clean', engine, if_exists='replace', index=False)
d_trips.to_sql('trips_clean', engine, if_exists='replace', index=False)
d_sensor_events.to_sql('sensor_events_clean', engine, if_exists='replace', index=False)
d_maintenance.to_sql('maintenance_logs_clean', engine, if_exists='replace', index=False)

70

In [80]:
df_vehicles = pd.read_sql("SELECT * FROM vehicles_clean", engine)
df_drivers = pd.read_sql("SELECT * FROM drivers_clean", engine)
df_trips = pd.read_sql("SELECT * FROM trips_clean", engine)
df_sensor_events = pd.read_sql("SELECT * FROM sensor_events_clean", engine)

print(f"Loaded: {len(df_trips)} trips,{len(df_sensor_events)} sensor events")


Loaded: 1470 trips,1702 sensor events


In [81]:
print(df_trips.columns)
"""
Feature Engineering - Telematics Project
-------------------------------------------
Reads cleaned tables from MySQL, calculates derived metrics needed for
driver risk scoring and dashboard analysis, and writes results back to MySQL.

Metrics calculated:
1. Fuel efficiency (km/l) per trip and per driver
2. Idle time % per vehicle
3. Harsh-event frequency per driver (normalized per 100 km)
4. Average speed per trip
5. Driver-level summary table (feeds into risk scoring)
"""

Index(['trip_id', 'vehicle_id', 'driver_id', 'start_time', 'end_time',
       'start_lat', 'start_long', 'end_lat', 'end_long', 'distance_km',
       'fuel_used_liters', 'idle_minutes'],
      dtype='str')


'\nFeature Engineering - Telematics Project\n-------------------------------------------\nReads cleaned tables from MySQL, calculates derived metrics needed for\ndriver risk scoring and dashboard analysis, and writes results back to MySQL.\n\nMetrics calculated:\n1. Fuel efficiency (km/l) per trip and per driver\n2. Idle time % per vehicle\n3. Harsh-event frequency per driver (normalized per 100 km)\n4. Average speed per trip\n5. Driver-level summary table (feeds into risk scoring)\n'

In [82]:

# ---------------------------
# 3. TRIP-LEVEL FEATURES
# ---------------------------
print(df_trips.head(3))
df_trips['start_time'] = pd.to_datetime(df_trips['start_time'])
df_trips['end_time'] = pd.to_datetime(df_trips['end_time'])

# Duration in hours
df_trips['duration_hours'] = (df_trips['end_time'] - df_trips['start_time']).dt.total_seconds() / 3600

# Average speed per trip (km/h)
df_trips['avg_speed_kmph'] = df_trips['distance_km'] / df_trips['duration_hours']

# Fuel efficiency per trip (km/l) - avoid divide by zero
df_trips['fuel_efficiency_kmpl'] = df_trips.apply(
    lambda row: row['distance_km'] / row['fuel_used_liters'] if row['fuel_used_liters'] > 0 else None,
    axis=1
)

# Idle time as % of total trip duration
df_trips['idle_pct'] = (df_trips['idle_minutes'] / (df_trips['duration_hours'] * 60)) * 100

print("\nTrip-level features calculated:")
print(df_trips[['trip_id', 'avg_speed_kmph', 'fuel_efficiency_kmpl', 'idle_pct']].head())
print(df_trips.columns.to_list())

   trip_id  vehicle_id  driver_id          start_time            end_time  \
0        1          20          7 2026-06-07 03:35:57 2026-06-07 04:34:13   
1        2          16          1 2026-06-03 04:07:43 2026-06-03 04:34:54   
2        3           3          3 2026-08-02 08:55:34 2026-08-02 10:27:38   

   start_lat  start_long    end_lat   end_long  distance_km  fuel_used_liters  \
0  12.991227   80.393267  13.196871  80.383806        28.50              2.34   
1  13.185057   80.360339  13.030677  80.151220        19.58              1.38   
2  12.966271   80.341133  13.154547  80.170265        49.33              4.36   

   idle_minutes  
0             7  
1             1  
2            28  

Trip-level features calculated:
   trip_id  avg_speed_kmph  fuel_efficiency_kmpl   idle_pct
0        1       29.347826             12.179487  12.013730
1        2       43.217658             14.188406   3.678725
2        3       32.148443             11.314220  30.412744
3        4       33.1

In [ ]:
# ---------------------------
# 4. HARSH EVENT COUNT PER TRIP
# ---------------------------
# print(df_sensor_events.head(10))
harsh_events_per_trip = df_sensor_events.groupby('trip_id').size().reset_index(name='harsh_event_count')
# print(harsh_events_per_trip.head(10))
print(df_trips.columns.to_list())
df_trips = df_trips.merge(harsh_events_per_trip, on='trip_id', how='left')
df_trips['harsh_event_count'] = df_trips['harsh_event_count'].fillna(0)
# Harsh events normalized per 100 km driven
df_trips['harsh_events_per_100km'] = (df_trips['harsh_event_count'] / df_trips['distance_km']) * 100
print(df_trips.columns.to_list())

['trip_id', 'vehicle_id', 'driver_id', 'start_time', 'end_time', 'start_lat', 'start_long', 'end_lat', 'end_long', 'distance_km', 'fuel_used_liters', 'idle_minutes', 'duration_hours', 'avg_speed_kmph', 'fuel_efficiency_kmpl', 'idle_pct']


In [86]:
# ---------------------------
# 5. DRIVER-LEVEL AGGREGATION (feeds into risk scoring)
# ---------------------------
driver_summary = df_trips.groupby('driver_id').agg(
    total_trips=('trip_id', 'count'),
    total_distance_km=('distance_km', 'sum'),
    total_fuel_liters=('fuel_used_liters', 'sum'),
    avg_fuel_efficiency_kmpl=('fuel_efficiency_kmpl', 'mean'),
    avg_idle_pct=('idle_pct', 'mean'),
    total_harsh_events=('harsh_event_count', 'sum'),
    avg_speed_kmph=('avg_speed_kmph', 'mean')
).reset_index()
# print(driver_summary.head(5))
# Harsh events per 100km at driver level (more meaningful than per-trip)
driver_summary['harsh_events_per_100km'] = (
    driver_summary['total_harsh_events'] / driver_summary['total_distance_km']
) * 100

# Merge in driver names for readability
driver_summary = driver_summary.merge(df_drivers[['driver_id', 'driver_name']], on='driver_id', how='left')

print("\nDriver-level summary (first 5 rows):")
print(driver_summary.head(5))



Driver-level summary (first 5 rows):
   driver_id  total_trips  total_distance_km  total_fuel_liters  \
0          1           37             911.28              82.50   
1          2           58            1444.27             124.00   
2          3           49            1291.54             116.94   
3          4           62            1553.13             137.26   
4          5           46            1128.96             101.54   

   avg_fuel_efficiency_kmpl  avg_idle_pct  total_harsh_events  avg_speed_kmph  \
0                 11.145159     24.690872                36.0       33.462249   
1                 11.749272     32.703985                65.0       33.200859   
2                 11.088962     22.293169                47.0       32.980535   
3                 11.499133     25.454880                85.0       33.691447   
4                 11.680595     23.800875                58.0       33.162660   

   harsh_events_per_100km        driver_name  
0                3.950487

In [87]:
# ---------------------------
# 6. VEHICLE-LEVEL AGGREGATION (idle %, efficiency by vehicle)
# ---------------------------
vehicle_summary = df_trips.groupby('vehicle_id').agg(
    total_trips=('trip_id', 'count'),
    total_distance_km=('distance_km', 'sum'),
    avg_fuel_efficiency_kmpl=('fuel_efficiency_kmpl', 'mean'),
    avg_idle_pct=('idle_pct', 'mean'),
    total_harsh_events=('harsh_event_count', 'sum')
).reset_index()

vehicle_summary = vehicle_summary.merge(
    df_vehicles[['vehicle_id', 'vehicle_number', 'vehicle_type']], on='vehicle_id', how='left'
)

print("\nVehicle-level summary (first 5 rows):")
print(vehicle_summary.head())



Vehicle-level summary (first 5 rows):
   vehicle_id  total_trips  total_distance_km  avg_fuel_efficiency_kmpl  \
0           1           64            1572.37                 11.574945   
1           2           52            1252.14                 11.248073   
2           3           59            1670.93                 11.706051   
3           4           60            1419.98                 11.702841   
4           5           58            1433.74                 11.570456   

   avg_idle_pct  total_harsh_events vehicle_number vehicle_type  
0     27.917300                83.0     TN91AB2824   Mini Truck  
1     27.291643                54.0     TN45AB5012          Van  
2     25.430219                76.0     TN23AB9935   Mini Truck  
3     22.558420                62.0     TN64AB1520   Mini Truck  
4     21.633253                67.0     TN37AB4811   Mini Truck  


In [88]:
# ---------------------------
# 7. WRITE FEATURE TABLES BACK TO MYSQL
# ---------------------------
df_trips.to_sql('trips_features', engine, if_exists='replace', index=False)
driver_summary.to_sql('driver_summary', engine, if_exists='replace', index=False)
vehicle_summary.to_sql('vehicle_summary', engine, if_exists='replace', index=False)

print("\nFeature tables written to MySQL: trips_features, driver_summary, vehicle_summary")

engine.dispose()
print("Connection closed.")


Feature tables written to MySQL: trips_features, driver_summary, vehicle_summary
Connection closed.


In [89]:
def normalize(series, higher_is_worse=True):
    min_val, max_val = series.min(), series.max()
    if higher_is_worse:
        return (series - min_val) / (max_val - min_val) * 100
    else:
        return (max_val - series) / (max_val - min_val) * 100  # invert for efficiency (lower = worse)

driver_summary['idle_score'] = normalize(driver_summary['avg_idle_pct'], higher_is_worse=True)
driver_summary['harsh_score'] = normalize(driver_summary['harsh_events_per_100km'], higher_is_worse=True)
driver_summary['efficiency_score'] = normalize(driver_summary['avg_fuel_efficiency_kmpl'], higher_is_worse=False)
# Weights should sum to 1.0 - adjust based on what the business cares about most
WEIGHT_HARSH = 0.5      # safety/accident risk - weighted highest
WEIGHT_IDLE = 0.3       # fuel waste
WEIGHT_EFFICIENCY = 0.2 # cost efficiency

driver_summary['risk_score'] = (
    driver_summary['harsh_score'] * WEIGHT_HARSH +
    driver_summary['idle_score'] * WEIGHT_IDLE +
    driver_summary['efficiency_score'] * WEIGHT_EFFICIENCY
)

driver_summary = driver_summary.sort_values('risk_score', ascending=False)
def risk_tier(score):
    if score >= 70:
        return 'High Risk'
    elif score >= 40:
        return 'Medium Risk'
    else:
        return 'Low Risk'

driver_summary['risk_tier'] = driver_summary['risk_score'].apply(risk_tier)
driver_summary.to_sql('driver_risk_scores', engine, if_exists='replace', index=False)

30